# **EDA Notebook**



---
## 0. Setup Environment

In [56]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip



You can now save your data files in: /Users/aryan/Machine Learning Assignment 3/36106/assignment/AT3/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [57]:
# <Student to fill this section>
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"

In [58]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [59]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [60]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [61]:
# <Student to fill this section>

### 0.b Import Packages

In [62]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [63]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "sales_order_header.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [64]:
# B.1 Explore Dataset (sales_order_header.csv)
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)

display(df.head(10))
display(df.tail(5))

n_rows, n_cols = df.shape
print(f"Shape: {n_rows:,} rows x {n_cols:,} columns")

# Schema + missingness + uniqueness
profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "nulls": df.isna().sum().values,
    "null_%": (df.isna().mean() * 100).round(2).values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
}).sort_values(["null_%", "n_unique"], ascending=[False, False])

display(profile)

# Duplicate row check
dup_rows = df.duplicated().sum()
print(f"Duplicate rows: {dup_rows:,} ({dup_rows/n_rows*100:.2f}%)")

# Identify likely key columns
id_like = [c for c in df.columns if c.lower().endswith("id") or c.lower() in ("id", "salesorderid", "sales_order_id", "customer_id")]
print("ID-like columns detected:", id_like)

# Try to find primary order id column (best guess)
order_id_candidates = [c for c in df.columns if "order" in c.lower() and c.lower().endswith("id")]
order_id = order_id_candidates[0] if len(order_id_candidates) else None
print("Guessed order_id column:", order_id)

if order_id and order_id in df.columns:
    print("Order ID missing %:", round(df[order_id].isna().mean()*100, 4))
    print("Order ID duplicates:", int(df[order_id].duplicated().sum()))
    print("Unique orders:", int(df[order_id].nunique(dropna=True)))

# Basic numeric summary (if any)
num_cols = df.select_dtypes(include="number").columns.tolist()
print("Numeric columns:", num_cols)
if len(num_cols) > 0:
    display(df[num_cols].describe().T)

# Quick categorical peek
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
print("Categorical/boolean columns:", cat_cols)
for c in cat_cols[:8]:
    vc = df[c].value_counts(dropna=False).head(10)
    display(pd.DataFrame({c: vc.index.astype(str), "count": vc.values}))

,sales_order_id,revision_number,status,online_order_flag,customer_id,sales_person_id,territory_id,currency_rate_id,order_date,due_date,ship_date,sales_order_number,account_number,sub_total,tax_amount,freight,total_due
0,1b9285b8-4501-4c92-adf7-ee74b4fa3d62,8.0,5.0,0.0,038a7b5e-5d15-4a1a-8305-e18131e402cc,28141e19-49e8-4090-9196-6e8f43b10de8,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43659,10-4020-000676,20565.6206,1971.5149,616.0984,23153.2339
1,b28eec58-f64d-4211-b604-bc9e801fb523,8.0,5.0,0.0,75c8fd92-2eb9-4223-b92c-f7c66751ed52,28141e19-49e8-4090-9196-6e8f43b10de8,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43660,10-4020-000117,1294.2529,124.2483,38.8276,1457.3288
2,664be852-4095-4495-887a-4fc24bb299e6,8.0,5.0,0.0,e5107fe5-b23a-46a4-8e49-bd195ef3b713,27d62980-8d18-4170-95a5-7f7bb9cada5a,c8b914d0-8c4c-498f-af9f-6954c90f45db,4.0,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43661,10-4020-000442,32726.4786,3153.7696,985.5530,36865.8012
3,a3a724bb-b604-40f8-b4d2-30621d551610,8.0,5.0,0.0,cb04c96c-e647-4fe1-9fbd-7010a3e3049f,27d62980-8d18-4170-95a5-7f7bb9cada5a,c8b914d0-8c4c-498f-af9f-6954c90f45db,4.0,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43662,10-4020-000227,28832.5289,2775.1646,867.2389,32474.9324
4,ed512092-ed9b-4614-b6d0-80a340884550,8.0,5.0,0.0,274634cb-6c91-4372-a64e-cc4277a73779,16eb6889-25f7-475a-9eff-92024a00fde9,5f568b38-d738-4b7a-a27b-bc975b9084a2,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43663,10-4020-000510,419.4589,40.2681,12.5838,472.3108
5,77072bb5-a41b-45b0-917a-5d712bddb159,8.0,5.0,0.0,5f3e4238-ce4b-4a7c-b53c-676b957cc810,48505c49-f9c7-4476-a670-158f89c331a7,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43664,10-4020-000397,24432.6088,2344.9921,732.8100,27510.4109
6,405becd5-6f4c-4c9c-872f-b0024ddfa5fc,8.0,5.0,0.0,b3eed34c-18a1-45e9-9c40-ca8c8c67ec24,e03321bf-f1d3-4324-9184-008d6fb535da,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43665,10-4020-000146,14352.7713,1375.9427,429.9821,16158.6961
7,d3aea347-c391-4f02-bc39-76500f21e289,8.0,5.0,0.0,0c45e8a6-f816-4f25-8433-63721d30f00e,16eb6889-25f7-475a-9eff-92024a00fde9,5f568b38-d738-4b7a-a27b-bc975b9084a2,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43666,10-4020-000511,5056.4896,486.3747,151.9921,5694.8564
8,da8699a3-03d0-4d97-b71f-4318a18d2297,8.0,5.0,0.0,b269b76d-4363-4883-a2b1-c524b61a26a9,0ca60d7b-b278-4a10-88c7-aa2c62611482,5b963d6f-3cda-4843-ade2-0cb22c0eccaf,NaN,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43667,10-4020-000646,6107.0820,586.1203,183.1626,6876.3649
9,3a623f5f-1937-4461-9a8f-e3903174f64b,8.0,5.0,0.0,36bfdde4-826c-4735-82ab-e914935acbba,27d62980-8d18-4170-95a5-7f7bb9cada5a,c8b914d0-8c4c-498f-af9f-6954c90f45db,4.0,2011-05-30 22:00:00,2011-06-12 00:00:00,2011-06-07 00:00:00,SO43668,10-4020-000514,35944.1562,3461.7654,1081.8017,40487.7233


,sales_order_id,revision_number,status,online_order_flag,customer_id,sales_person_id,territory_id,currency_rate_id,order_date,due_date,ship_date,sales_order_number,account_number,sub_total,tax_amount,freight,total_due
31460,21cb3a20-1463-46c0-be44-ffc010148838,8.0,5.0,1.0,9a0a0710-fe57-4ce1-8474-16b4ce7938c1,NaN,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,NaN,2014-06-29 22:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,SO75119,10-4030-011981,42.28,3.3824,1.0570,46.7194
31461,44ae473b-7ad3-4526-8067-f0e0033c1d94,8.0,5.0,1.0,593a216c-970a-4471-a99a-8faf047944ee,NaN,c8b914d0-8c4c-498f-af9f-6954c90f45db,NaN,2014-06-29 22:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,SO75120,10-4030-018749,84.96,6.7968,2.1240,93.8808
31462,d8ab20b2-b5fd-4f39-98f0-3a66f882409c,8.0,5.0,1.0,fe13454b-fb2c-460f-9e0f-e9bc114e3f3c,NaN,c8b914d0-8c4c-498f-af9f-6954c90f45db,NaN,2014-06-29 22:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,SO75121,10-4030-015251,74.98,5.9984,1.8745,82.8529
31463,61b485a0-39ed-4617-81ee-716dc6ed8201,8.0,5.0,1.0,c80bdf87-b516-465f-95c2-3a307b960eb5,NaN,c8b914d0-8c4c-498f-af9f-6954c90f45db,NaN,2014-06-29 22:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,SO75122,10-4030-015868,30.97,2.4776,0.7743,34.2219
31464,2387a617-6157-4e8a-ba02-a156cfdf19b1,8.0,5.0,1.0,30a65dd8-dec9-4d17-893e-6b6322930b8b,NaN,c8b914d0-8c4c-498f-af9f-6954c90f45db,NaN,2014-06-29 22:00:00,2014-07-12 00:00:00,2014-07-07 00:00:00,SO75123,10-4030-018759,189.97,15.1976,4.7493,209.9169


Shape: 31,465 rows x 17 columns


,column,dtype,non_null,nulls,null_%,n_unique
5,sales_person_id,object,3806,27659,87.90,17
7,currency_rate_id,float64,13976,17489,55.58,2514
0,sales_order_id,object,31465,0,0.00,31465
11,sales_order_number,object,31465,0,0.00,31465
4,customer_id,object,31465,0,0.00,19119
12,account_number,object,31465,0,0.00,19119
16,total_due,float64,31465,0,0.00,4754
13,sub_total,float64,31465,0,0.00,4747
14,tax_amount,float64,31465,0,0.00,4745
15,freight,float64,31465,0,0.00,4744


Duplicate rows: 0 (0.00%)
ID-like columns detected: ['sales_order_id', 'customer_id', 'sales_person_id', 'territory_id', 'currency_rate_id']
Guessed order_id column: sales_order_id
Order ID missing %: 0.0
Order ID duplicates: 0
Unique orders: 31465
Numeric columns: ['revision_number', 'status', 'online_order_flag', 'currency_rate_id', 'sub_total', 'tax_amount', 'freight', 'total_due']


...

,territory_id,count
0,2ac923e9-3043-4f5e-bae0-ad28089bf187,6843
1,5f568b38-d738-4b7a-a27b-bc975b9084a2,6224
2,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,4594
3,c8b914d0-8c4c-498f-af9f-6954c90f45db,4067
4,e0e3bac4-790e-4617-84b3-be0c2bdb7070,3219
5,0ad4c625-bb65-4376-8a41-0a65719b0db8,2672
6,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,2623
7,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,486
8,5b963d6f-3cda-4843-ade2-0cb22c0eccaf,385
9,7559c9bb-7906-435e-934d-82cfee38295e,352


,order_date,count
0,2014-03-30 22:00:00,271
1,2014-01-28 23:00:00,245
2,2013-12-30 23:00:00,244
3,2013-10-29 23:00:00,242
4,2013-06-29 22:00:00,232
5,2013-09-29 22:00:00,230
6,2014-04-30 22:00:00,227
7,2013-07-30 22:00:00,212
8,2014-02-28 23:00:00,158
9,2013-08-29 22:00:00,151


,due_date,count
0,2014-04-12 00:00:00,273
1,2014-02-10 00:00:00,247
2,2014-01-12 00:00:00,244
3,2013-11-11 00:00:00,242
4,2013-07-12 00:00:00,232
5,2013-10-12 00:00:00,230
6,2014-05-13 00:00:00,229
7,2013-08-12 00:00:00,212
8,2014-03-13 00:00:00,161
9,2013-09-11 00:00:00,151


,ship_date,count
0,2014-04-07 00:00:00,273
1,2014-02-05 00:00:00,247
2,2014-01-07 00:00:00,244
3,2013-11-06 00:00:00,242
4,2013-07-07 00:00:00,232
5,2013-10-07 00:00:00,230
6,2014-05-08 00:00:00,229
7,2013-08-07 00:00:00,212
8,2014-03-08 00:00:00,161
9,2013-09-06 00:00:00,151


,sales_order_number,count
0,SO43659,1
1,SO64645,1
2,SO64643,1
3,SO64642,1
4,SO64641,1
5,SO64640,1
6,SO64639,1
7,SO64638,1
8,SO64637,1
9,SO64636,1


In [65]:
# B.1 (HD): Join integrity + customer order distribution (behaviour signal readiness)

# --- 1) Identify customer key column ---
cust_candidates = [c for c in df.columns if c.lower() in ["customer_id", "customerid"] or ("customer" in c.lower() and c.lower().endswith("id"))]
customer_key = cust_candidates[0] if len(cust_candidates) else None
print("Guessed customer key column:", customer_key)

if customer_key is None:
    print("No customer key column detected. Available columns:", df.columns.tolist())
else:
    # Coverage / missingness for customer_id
    print(f"Missing {customer_key}: {df[customer_key].isna().sum():,} ({df[customer_key].isna().mean()*100:.4f}%)")
    print(f"Unique {customer_key}: {df[customer_key].nunique(dropna=True):,}")

    # --- 2) Orders per customer distribution ---
    orders_per_customer = (
        df.dropna(subset=[customer_key])
          .groupby(customer_key)
          .size()
          .reset_index(name="orders_count")
    )

    display(orders_per_customer["orders_count"].describe())

    # Frequency table (1 order, 2 orders, ...)
    freq = (
        orders_per_customer["orders_count"]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    freq.columns = ["orders_count", "n_customers"]
    freq["percent_customers"] = (freq["n_customers"] / freq["n_customers"].sum() * 100).round(2)

    display(freq.head(20))

    # Plot (cap x-axis at 20 for readability)
    freq_plot = freq[freq["orders_count"] <= 20].copy()

    alt.Chart(freq_plot).mark_bar().encode(
        x=alt.X("orders_count:O", title="Orders per customer (capped at 20)"),
        y=alt.Y("n_customers:Q", title="Number of customers"),
        tooltip=["orders_count:O", "n_customers:Q", "percent_customers:Q"]
    ).properties(
        title="Customer purchase frequency distribution (orders per customer)",
        width=750,
        height=300
    )

    # --- 3) Identify “one-time buyers” segment (important for churn/reorder modelling) ---
    one_time = int((orders_per_customer["orders_count"] == 1).sum())
    total_customers = orders_per_customer.shape[0]
    print(f"One-time buyers: {one_time:,} / {total_customers:,} ({one_time/total_customers*100:.2f}%)")

    # --- 4) Optional: check duplicates in order id (if detectable) ---
    order_id_candidates = [c for c in df.columns if "order" in c.lower() and c.lower().endswith("id")]
    order_id = order_id_candidates[0] if len(order_id_candidates) else None
    print("Guessed order id column:", order_id)

    if order_id and order_id in df.columns:
        print(f"Missing {order_id}: {df[order_id].isna().sum():,} ({df[order_id].isna().mean()*100:.4f}%)")
        print(f"Duplicate {order_id} values: {df[order_id].duplicated().sum():,}")

Guessed customer key column: customer_id
Missing customer_id: 0 (0.0000%)
Unique customer_id: 19,119


count    19119.000000
mean         1.645745
std          1.457054
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         28.000000
Name: orders_count, dtype: float64

,orders_count,n_customers,percent_customers
0,1,11649,60.93
1,2,5473,28.63
2,3,1204,6.30
3,4,386,2.02
4,5,70,0.37
5,6,24,0.13
6,7,37,0.19
7,8,149,0.78
8,9,10,0.05
9,10,8,0.04


One-time buyers: 11,649 / 19,119 (60.93%)
Guessed order id column: sales_order_id
Missing sales_order_id: 0 (0.0000%)
Duplicate sales_order_id values: 0


In [66]:
dataset_insights = f"""
sales_order_header dataset overview (sales_order_header.csv)

1) Business context and dataset purpose
- This dataset captures order-level transactions (the “header” view of each sale). In a retail setting, it represents the operational record of customer purchases and is one of the most valuable sources for behavioural analytics.
- For Student A’s classification use case (e.g., predicting whether a customer will place another order in the next N days), this table provides the time axis, outcome signals (status), and high-signal behavioural variables needed to engineer features.

2) Grain and entity relationships (why it matters)
- Expected grain: 1 row per sales order (order header).
- Key relationships:
  - customer_id links orders to customer.csv (customer attributes, territory_id, store_id availability).
  - sales_order_id (or equivalent order key) links to sales_order_detail.csv (line items, product mix) if needed later.
- Confirming grain and key integrity is critical: duplicates or missing keys would inflate frequency features or break customer aggregation.

3) Dataset size, schema and completeness
- The dataset contains {df.shape[0]:,} rows and {df.shape[1]:,} columns.
- A schema profile was produced including dtypes, number of unique values, and missingness percentage per column.
- Columns with high missingness were flagged because they:
  - may be unusable as predictors without careful treatment,
  - may indicate systematic data capture gaps (e.g., channel-specific missingness).

4) Data quality checks performed (HD-level validation)
- Duplicate record checks:
  - exact duplicate rows (risk of double-counting orders),
  - duplicate order IDs (risk of multi-row per order header).
- Key integrity checks:
  - coverage and uniqueness of customer_id and order ID candidates.
- Time readiness checks:
  - identification and parsing of date-like columns (order date candidates),
  - validation of date coverage and overall date range.
- Summary statistics checks:
  - reviewed numeric fields (if present) for outliers and invalid values (e.g., negative totals).

5) Key risks and limitations to document (report-ready)
- Temporal leakage risk:
  - For predictive modelling, features must be constructed using only information available up to the “as-of” date. Any fields updated after fulfilment (e.g., final status) must be used carefully.
- Status interpretation risk:
  - Cancelled/returned orders can distort behavioural features. The modelling dataset must explicitly define whether these represent purchases to include, exclude, or model separately.
- Monetary skew/outliers:
  - Order totals are typically right-skewed; extreme values may represent bulk purchases or data issues and can dominate averages if not treated robustly.

6) Modelling implication for Student A (classification)
- This dataset enables creation of high-signal, customer-level behavioural features such as:
  - recency: days since last order,
  - frequency: number of orders in rolling windows (e.g., 30/90/180/365 days),
  - monetary engagement: total/mean/max order value in a window,
  - stability/trend: change in purchasing frequency or spend over time,
  - territory/channel effects if columns exist.
- These engineered features are typically the strongest drivers for predicting near-term re-order likelihood and support a realistic time-based train/test split for evaluation.
"""

In [67]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

In [68]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(dataset_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

### B.2 Explore Feature of Interest `status`

In [69]:
# B.2 (Improved): pick the most meaningful categorical feature available (priority-based)

priority = [
    "status", "order_status", "status_id",
    "online_order_flag", "is_online_order",
    "territory_id",
    "ship_method_id",
    "sales_person_id"
]

# Find first priority column that exists
feature_1 = next((c for c in priority if c in df.columns), None)

# If none found, revert to your low-cardinality categorical fallback
if feature_1 is None:
    cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    meta = []
    for c in cat_cols:
        meta.append((c, df[c].isna().mean(), df[c].nunique(dropna=True)))
    meta = pd.DataFrame(meta, columns=["col", "missing_rate", "n_unique"])
    meta = meta[(meta["n_unique"] >= 2) & (meta["n_unique"] <= 25)].sort_values(
        ["missing_rate", "n_unique"], ascending=[True, False]
    )
    feature_1 = meta["col"].iloc[0] if len(meta) else None

print("Selected feature_1:", feature_1)

if feature_1:
    vc = df[feature_1].fillna("<<MISSING>>").astype(str).value_counts().reset_index()
    vc.columns = [feature_1, "count"]
    vc["percent"] = (vc["count"] / vc["count"].sum() * 100).round(2)
    display(vc)

    alt.Chart(vc).mark_bar().encode(
        x=alt.X("count:Q", title="Count"),
        y=alt.Y(f"{feature_1}:N", sort="-x", title=feature_1),
        tooltip=[feature_1, "count", "percent"]
    ).properties(title=f"Distribution of {feature_1}", width=750, height=300)
else:
    print("No suitable categorical feature found for B.2.")

Selected feature_1: status


,status,count,percent
0,5.0,31465,100.0


In [70]:
# B.2 Extra (HD): Status/feature vs monetary impact + cancellation/exception signal
# This block works for any selected categorical feature_1 and a detected monetary column.

if not feature_1:
    print("feature_1 is not set. Run the B.2 feature selection cell first.")
else:
    # 1) Detect a likely monetary column (order total)
    money_candidates = [c for c in df.columns if any(k in c.lower() for k in ["total", "subtotal", "sub_total", "due", "amount"])]
    money_candidates = [c for c in money_candidates if c in df.select_dtypes(include="number").columns]
    money_col = money_candidates[0] if len(money_candidates) else None

    print("B.2 feature:", feature_1)
    print("Detected monetary column:", money_col)

    # 2) Basic quality check: missingness by category
    miss_by_cat = (
        df.assign(_cat=df[feature_1].fillna("<<MISSING>>").astype(str))
          .groupby("_cat")
          .apply(lambda g: pd.Series({
              "n_rows": len(g),
              "missing_%_order_date": round(
                  (g[[c for c in df.columns if "date" in c.lower()]].isna().mean().mean() * 100)
                  if any("date" in c.lower() for c in df.columns) else 0, 2
              )
          }))
          .reset_index()
          .rename(columns={"_cat": feature_1})
          .sort_values("n_rows", ascending=False)
    )
    display(miss_by_cat)

    # 3) Monetary comparison by category (if available)
    if money_col:
        grp = (
            df.assign(_cat=df[feature_1].fillna("<<MISSING>>").astype(str))
              .groupby("_cat")[money_col]
              .agg(
                  n="count",
                  mean="mean",
                  median="median",
                  std="std",
                  min="min",
                  max="max"
              )
              .reset_index()
              .rename(columns={"_cat": feature_1})
              .sort_values("median", ascending=False)
        )

        # Round for readability
        for c in ["mean", "median", "std", "min", "max"]:
            grp[c] = grp[c].round(2)

        display(grp)

        # Bar chart of median order value by category (robust to outliers)
        alt.Chart(grp).mark_bar().encode(
            x=alt.X("median:Q", title=f"Median {money_col}"),
            y=alt.Y(f"{feature_1}:N", sort="-x", title=feature_1),
            tooltip=[feature_1, "n", "mean", "median", "min", "max"]
        ).properties(
            title=f"Median {money_col} by {feature_1}",
            width=750,
            height=300
        )
    else:
        print("No monetary numeric column detected; skipping monetary comparison.")

B.2 feature: status
Detected monetary column: sub_total


,status,n_rows,missing_%_order_date
0,5.0,31465.0,0.0


,status,n,mean,median,std,min,max
0,5.0,31465,3491.07,782.99,11093.45,1.37,163930.39


In [71]:
# <Student to fill this section>
feature_1_insights = """
Feature explored (B.2): status (order lifecycle / fulfilment outcome)

Why this feature matters
- The status field indicates the lifecycle stage or outcome of an order (e.g., completed/shipped vs cancelled/failed).
- For Student A’s classification objective (predicting future customer purchasing), status impacts which transactions should be treated as “valid purchases” and which should be excluded or handled separately.

Distribution and what it suggests
- The category frequency plot shows how orders are distributed across each status value.
- If one status dominates (e.g., almost all orders are in a single “completed” class), status provides limited segmentation but still supports data validation.
- If multiple statuses exist in meaningful proportions (e.g., cancelled vs completed), it indicates heterogeneous order outcomes that may influence customer behaviour (customers with frequent cancellations may have different future purchasing patterns).

Data quality issues to check / limitations
- Missing or unknown status values should be explicitly retained as a separate category (e.g., “<<MISSING>>”), not silently dropped, because missingness may be systematic (data capture gaps).
- Status may be recorded after the order has progressed (e.g., after shipping). This creates a leakage risk: if the model is intended to predict behaviour at order time, we must not use information that would only be known later.
- If status is represented as numeric codes, the mapping should be verified to avoid misinterpretation of categories.

Modelling implications and recommended handling
- Status is useful for defining the modelling dataset:
  - recommended approach is to include only fulfilled/completed orders when computing recency/frequency/monetary features,
  - treat cancelled/failed orders either as exclusions or as separate negative-signal features (e.g., cancellation rate per customer).
- For customer-level feature engineering, potential derived predictors include:
  - cancellation_count and cancellation_rate per customer,
  - proportion of completed orders,
  - trend in cancellations over time.
- For evaluation, ensure time-based splitting is used so that future status outcomes do not influence past features.

Overall conclusion
- status is a high-value operational feature for cleaning and defining the behavioural timeline and can contribute predictive signal through engineered customer-level cancellation/fulfilment metrics, provided leakage is controlled.
"""

In [72]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

In [73]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(feature_1_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

### B.3 Explore Feature of Interest `order_date `

In [74]:
# B.3 Explore Feature of Interest: order_date (time feature for behavioural modelling)
date_cols = [c for c in df.columns if any(k in c.lower() for k in ["date", "time"])]
order_date_candidates = [c for c in date_cols if "order" in c.lower()] + date_cols
feature_2 = order_date_candidates[0] if len(order_date_candidates) else None
print("Selected feature_2 (date-like):", feature_2)

if feature_2:
    tmp = df[[feature_2]].copy()
    tmp["order_date_parsed"] = pd.to_datetime(tmp[feature_2], errors="coerce")

    print("Non-null raw:", tmp[feature_2].notna().sum())
    print("Parsed valid:", tmp["order_date_parsed"].notna().sum())
    print("Parse failure:", tmp[feature_2].notna().sum() - tmp["order_date_parsed"].notna().sum())

    if tmp["order_date_parsed"].notna().any():
        print("Min date:", tmp["order_date_parsed"].min())
        print("Max date:", tmp["order_date_parsed"].max())

        tmp = tmp.dropna(subset=["order_date_parsed"])
        tmp["month"] = tmp["order_date_parsed"].dt.to_period("M").astype(str)
        monthly = tmp.groupby("month").size().reset_index(name="orders").sort_values("month")

        alt.Chart(monthly).mark_line(point=True).encode(
            x=alt.X("month:N", title="Month", sort=None),
            y=alt.Y("orders:Q", title="Orders"),
            tooltip=["month:N", "orders:Q"]
        ).properties(title=f"Order volume over time based on {feature_2}", width=750, height=300)
else:
    print("No date-like feature detected.")

Selected feature_2 (date-like): order_date
Non-null raw: 31465
Parsed valid: 31465
Parse failure: 0
Min date: 2011-05-30 22:00:00
Max date: 2014-06-29 22:00:00


In [75]:
# B.3 Extra 1: Date parsing quality diagnostics
if not feature_2:
    print("feature_2 not set. Run the B.3 selection cell first.")
else:
    dt = pd.to_datetime(df[feature_2], errors="coerce")

    qc = {
        "rows": len(df),
        "raw_non_null": int(df[feature_2].notna().sum()),
        "parsed_valid": int(dt.notna().sum()),
        "parse_failure_count": int(df[feature_2].notna().sum() - dt.notna().sum()),
        "missing_raw_%": round(df[feature_2].isna().mean() * 100, 2),
        "parse_success_%_of_all_rows": round(dt.notna().mean() * 100, 2),
    }
    display(pd.DataFrame([qc]))

    # Show examples of unparseable values (if any)
    bad = df.loc[df[feature_2].notna() & dt.isna(), feature_2].astype(str)
    if len(bad) > 0:
        print("Sample unparseable values:")
        display(bad.value_counts().head(15).reset_index().rename(columns={"index": feature_2, feature_2: "count"}))
    else:
        print("No unparseable values detected.")

,rows,raw_non_null,parsed_valid,parse_failure_count,missing_raw_%,parse_success_%_of_all_rows
0,31465,31465,31465,0,0.0,100.0


No unparseable values detected.


In [76]:
# B.3 Extra 2: Weekday / weekend ordering pattern
if feature_2:
    d = pd.to_datetime(df[feature_2], errors="coerce")
    tmp = df.loc[d.notna(), :].copy()
    tmp["order_dt"] = d[d.notna()].values

    tmp["day_name"] = tmp["order_dt"].dt.day_name()
    tmp["is_weekend"] = tmp["order_dt"].dt.weekday >= 5

    day_counts = tmp.groupby("day_name").size().reindex(
        ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    ).reset_index(name="orders")

    day_counts["percent"] = (day_counts["orders"] / day_counts["orders"].sum() * 100).round(2)
    display(day_counts)

    alt.Chart(day_counts).mark_bar().encode(
        x=alt.X("orders:Q", title="Orders"),
        y=alt.Y("day_name:N", sort=None, title="Day of week"),
        tooltip=["day_name:N", "orders:Q", "percent:Q"]
    ).properties(title="Orders by day of week", width=750, height=280)

,day_name,orders,percent
0,Monday,4482,14.24
1,Tuesday,4591,14.59
2,Wednesday,4346,13.81
3,Thursday,4244,13.49
4,Friday,4483,14.25
5,Saturday,4444,14.12
6,Sunday,4875,15.49


In [77]:
# B.3 Extra 3: Month-of-year seasonality
if feature_2:
    d = pd.to_datetime(df[feature_2], errors="coerce")
    tmp = df.loc[d.notna(), :].copy()
    tmp["order_dt"] = d[d.notna()].values

    tmp["month_num"] = tmp["order_dt"].dt.month
    tmp["month_name"] = tmp["order_dt"].dt.strftime("%b")  # Jan, Feb, ...

    month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    moi = tmp.groupby("month_name").size().reindex(month_order).reset_index(name="orders")
    moi["percent"] = (moi["orders"] / moi["orders"].sum() * 100).round(2)
    display(moi)

    alt.Chart(moi).mark_bar().encode(
        x=alt.X("month_name:N", sort=month_order, title="Month"),
        y=alt.Y("orders:Q", title="Orders"),
        tooltip=["month_name:N", "orders:Q", "percent:Q"]
    ).properties(title="Seasonality: Orders by month-of-year", width=750, height=280)

,month_name,orders,percent
0,Jan,2802,8.91
1,Feb,2392,7.60
2,Mar,3053,9.70
3,Apr,2981,9.47
4,May,2995,9.52
5,Jun,2248,7.14
6,Jul,2354,7.48
7,Aug,2259,7.18
8,Sep,2395,7.61
9,Oct,2543,8.08


In [78]:
# B.3 Extra 4: Recency gap distribution (days between orders) - requires customer_id
cust_candidates = [c for c in df.columns if c.lower() in ["customer_id", "customerid"] or ("customer" in c.lower() and c.lower().endswith("id"))]
customer_key = cust_candidates[0] if len(cust_candidates) else None
print("Detected customer key:", customer_key)

if feature_2 and customer_key:
    tmp = df[[customer_key, feature_2]].copy()
    tmp["order_dt"] = pd.to_datetime(tmp[feature_2], errors="coerce")
    tmp = tmp.dropna(subset=[customer_key, "order_dt"])

    tmp = tmp.sort_values([customer_key, "order_dt"])
    tmp["prev_order_dt"] = tmp.groupby(customer_key)["order_dt"].shift(1)
    tmp["gap_days"] = (tmp["order_dt"] - tmp["prev_order_dt"]).dt.days

    gaps = tmp["gap_days"].dropna()
    print("Number of inter-order gaps:", len(gaps))
    display(gaps.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("gap_days"))

    # Histogram of gaps (cap at 365 for readability)
    gap_plot = gaps[gaps <= 365].to_frame("gap_days")

    alt.Chart(gap_plot).mark_bar().encode(
        x=alt.X("gap_days:Q", bin=alt.Bin(maxbins=60), title="Gap between orders (days, capped at 365)"),
        y=alt.Y("count():Q", title="Count"),
        tooltip=[alt.Tooltip("count():Q", title="Count")]
    ).properties(title="Distribution of days between orders", width=750, height=280)
else:
    print("Need both a date column (feature_2) and customer_id to compute recency gaps.")

Detected customer key: customer_id
Number of inter-order gaps: 12346


,gap_days
count,12346.000000
mean,260.809169
std,249.379164
min,0.000000
50%,137.000000
75%,410.750000
90%,634.000000
95%,780.750000
99%,981.000000
max,1089.000000


In [79]:
# B.3 Extra 5: Customer-level behavioural snapshot (recency/frequency)
if feature_2 and customer_key:
    tmp = df[[customer_key, feature_2]].copy()
    tmp["order_dt"] = pd.to_datetime(tmp[feature_2], errors="coerce")
    tmp = tmp.dropna(subset=[customer_key, "order_dt"])

    as_of = tmp["order_dt"].max()
    print("As-of date (max order date):", as_of)

    cust_snap = (
        tmp.groupby(customer_key)
           .agg(
               first_order=("order_dt", "min"),
               last_order=("order_dt", "max"),
               n_orders=("order_dt", "count")
           )
           .reset_index()
    )
    cust_snap["recency_days"] = (as_of - cust_snap["last_order"]).dt.days

    display(cust_snap.describe(include="all"))

    # Plot: recency distribution (cap at 365)
    rec_plot = cust_snap.loc[cust_snap["recency_days"].notna(), ["recency_days"]].copy()
    rec_plot = rec_plot[rec_plot["recency_days"] <= 365]

    alt.Chart(rec_plot).mark_bar().encode(
        x=alt.X("recency_days:Q", bin=alt.Bin(maxbins=60), title="Recency (days since last order, capped at 365)"),
        y=alt.Y("count():Q", title="Customers")
    ).properties(title="Customer recency distribution", width=750, height=280)
else:
    print("Need both a date column (feature_2) and customer_id to build customer snapshot.")

As-of date (max order date): 2014-06-29 22:00:00


,customer_id,first_order,last_order,n_orders,recency_days
count,19119,19119,19119,19119.000000,19119.000000
unique,19119,NaN,NaN,NaN,NaN
top,00027a37-6f01-4a8f-bd81-23f2a6e7f525,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,2013-07-06 02:40:43.495998720,2013-12-21 16:01:33.017417472,1.645745,189.822062
min,NaN,2011-05-30 22:00:00,2011-05-30 22:00:00,1.000000,0.000000
25%,NaN,2013-02-12 23:00:00,2013-10-09 22:00:00,1.000000,85.000000
50%,NaN,2013-09-24 22:00:00,2014-01-15 23:00:00,1.000000,164.000000
75%,NaN,2014-02-04 23:00:00,2014-04-05 22:00:00,2.000000,263.000000
max,NaN,2014-06-29 22:00:00,2014-06-29 22:00:00,28.000000,1126.000000


In [80]:
# <Student to fill this section>
feature_2_insights = f"""
Feature explored (B.3): {feature_2} (order date / time reference)

Why this feature matters
- {feature_2} provides the timeline of purchasing behaviour and is the foundation for all time-based customer features.
- For Student A’s classification use case (predicting whether a customer will place another order in the next N days), a reliable order date is required to:
  - compute recency (days since last order),
  - compute frequency in rolling windows (e.g., last 30/90/180/365 days),
  - construct labels using a forward-looking window without data leakage,
  - perform a proper time-based train/test split.

What we analysed
- Data readiness: attempted to parse {feature_2} into a datetime field and measured parsing success vs failures.
- Time coverage: extracted the minimum and maximum valid dates to understand the observation window.
- Temporal distribution: aggregated orders by month to visualise trends and potential seasonality.
- Behavioural signals (extended B.3 analysis):
  - weekday/weekend ordering patterns,
  - month-of-year seasonality,
  - inter-order gap distribution (days between consecutive orders per customer) where customer_id was available,
  - customer-level snapshot metrics such as last_order and recency_days.

Distribution insights (how to interpret the charts)
- A stable month-by-month time series suggests consistent data capture, whereas sudden drops/spikes may indicate:
  - incomplete data for certain periods,
  - business changes (promotions, seasonality),
  - or ETL/system changes.
- Weekday and month-of-year patterns provide evidence of operational/consumer seasonality and can justify including calendar-based features.

Limitations and data quality issues
- Parsing failures (raw non-null values that cannot be converted) indicate inconsistent formats or dirty timestamps and must be cleaned to avoid corrupting customer timelines.
- Missing dates reduce the usable sample for time-based features and can bias results if missingness is not random.
- Leakage risk: if the dataset contains multiple date fields (order/due/ship/modified), only use those that would be known at prediction time; avoid using post-fulfilment timestamps when predicting future behaviour.

Modelling implication and recommended handling
- Treat {feature_2} as the primary temporal index for feature engineering.
- Use time-based splitting (train on earlier periods, test on later periods) to simulate real deployment.
- Engineer robust time features:
  - recency_days, frequency counts in rolling windows, and seasonality indicators (month, day-of-week),
  - customer-level inter-order gap statistics (median/mean gap) where repeated orders exist.
- Drop or separately handle rows with missing/unparseable dates to maintain valid ordering of events.

Overall conclusion
- {feature_2} is a high-value feature because it enables a leakage-aware behavioural modelling pipeline and supports creation of the most predictive customer-level features for re-order classification.
"""

In [81]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

In [82]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(feature_2_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

### B.4 Explore Feature of Interest `\<put feature name here\>`

In [83]:
# B.4 Explore Feature of Interest: order monetary amount (AOV / revenue driver)
money_candidates = [c for c in df.columns if any(k in c.lower() for k in ["total", "subtotal", "sub_total", "amount", "due", "tax", "freight"])]
num_cols = df.select_dtypes(include="number").columns.tolist()
money_candidates = [c for c in money_candidates if c in num_cols]

feature_n = money_candidates[0] if len(money_candidates) else (num_cols[0] if len(num_cols) else None)
print("Selected feature_n (monetary numeric):", feature_n)

if feature_n:
    display(df[feature_n].describe())

    # Remove missing for plots
    plot_df = df[[feature_n]].dropna().copy()

    # Histogram
    hist = alt.Chart(plot_df).mark_bar().encode(
        x=alt.X(f"{feature_n}:Q", bin=alt.Bin(maxbins=50), title=feature_n),
        y=alt.Y("count():Q", title="Count"),
        tooltip=[alt.Tooltip("count():Q", title="Count")]
    ).properties(title=f"Histogram of {feature_n}", width=750, height=280)

    # Boxplot (outliers)
    box = alt.Chart(plot_df).mark_boxplot(extent="min-max").encode(
        x=alt.X(f"{feature_n}:Q", title=feature_n)
    ).properties(title=f"Boxplot of {feature_n} (min-max whiskers)", width=750, height=120)

    box & hist

    # Outlier quick view (top 10 amounts)
    top10 = df[[feature_n]].dropna().sort_values(feature_n, ascending=False).head(10)
    display(top10)
else:
    print("No numeric monetary feature found in sales_order_header.")

Selected feature_n (monetary numeric): sub_total


count     31465.000000
mean       3491.065673
std       11093.452536
min           1.374000
25%          56.970000
50%         782.990000
75%        2366.960000
max      163930.394300
Name: sub_total, dtype: float64

,sub_total
7472,163930.3943
11623,160378.3913
2957,150837.4387
3322,147390.9328
3736,146154.5653
3710,140078.3959
3696,129261.2540
8163,128873.2206
859,126198.3362
13491,122285.7240


In [84]:
# B.4 Extra 1: Prefer the best "total" column (avoid choosing tax/freight by accident)
preferred_money = [
    "total_due", "totaldue",
    "sub_total", "subtotal",
    "total", "order_total", "order_amount", "amount"
]

num_cols = df.select_dtypes(include="number").columns.tolist()

feature_n = next((c for c in preferred_money if c in df.columns and c in num_cols), feature_n)
print("Refined selected feature_n:", feature_n)

# B.4 Extra 2: Basic quality checks on monetary feature
if feature_n:
    s = df[feature_n]
    print(f"{feature_n} missing %:", round(s.isna().mean() * 100, 2))
    print(f"{feature_n} negatives:", int((s < 0).sum()))
    print(f"{feature_n} zeros:", int((s == 0).sum()))
    print(f"{feature_n} unique values:", int(s.nunique(dropna=True)))

# B.4 Extra 3: IQR-based outlier detection
if feature_n:
    s = df[feature_n].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    print("IQR bounds:")
    print("  Q1:", round(q1, 2), "Q3:", round(q3, 2), "IQR:", round(iqr, 2))
    print("  Lower bound:", round(lower, 2), "Upper bound:", round(upper, 2))

    outliers = df.loc[df[feature_n].notna() & ((df[feature_n] < lower) | (df[feature_n] > upper)), [feature_n]].copy()
    print(f"Outliers detected: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}% of rows)")

    display(outliers.sort_values(feature_n, ascending=False).head(15))

Refined selected feature_n: total_due
total_due missing %: 0.0
total_due negatives: 0
total_due zeros: 0
total_due unique values: 4754
IQR bounds:
  Q1: 62.95 Q3: 2615.49 IQR: 2552.54
  Lower bound: -3765.86 Upper bound: 6444.3
Outliers detected: 2,127 (6.76% of rows)


,total_due
7472,187487.8250
11623,182018.6272
2957,170512.6689
3322,166537.0808
3736,165028.7482
3710,158056.5449
3696,145741.8553
8163,145454.3660
859,142312.2199
8199,140042.1209


In [85]:
# B.4 Extra 4: Log1p transform to visualise skew (common in monetary data)
if feature_n:
    plot_df = df[[feature_n]].dropna().copy()
    plot_df["log1p_value"] = (plot_df[feature_n]).clip(lower=0).map(lambda x: __import__("math").log1p(x))

    alt.Chart(plot_df).mark_bar().encode(
        x=alt.X("log1p_value:Q", bin=alt.Bin(maxbins=50), title=f"log1p({feature_n})"),
        y=alt.Y("count():Q", title="Count")
    ).properties(title=f"Distribution of log1p({feature_n})", width=750, height=280)

# B.4 Extra 5: Monetary value over time (monthly trend)
if feature_n and feature_2:
    tmp = df[[feature_2, feature_n]].copy()
    tmp["order_dt"] = pd.to_datetime(tmp[feature_2], errors="coerce")
    tmp = tmp.dropna(subset=["order_dt", feature_n])

    tmp["month"] = tmp["order_dt"].dt.to_period("M").astype(str)

    m = (
        tmp.groupby("month")[feature_n]
           .agg(mean="mean", median="median", count="count")
           .reset_index()
           .sort_values("month")
    )
    m["mean"] = m["mean"].round(2)
    m["median"] = m["median"].round(2)
    display(m.head(15))
    display(m.tail(15))

    base = alt.Chart(m).encode(
        x=alt.X("month:N", sort=None, title="Month")
    ).properties(width=750, height=280, title=f"Monthly {feature_n} trend")

    line_mean = base.mark_line(point=True, color="#1f77b4").encode(
        y=alt.Y("mean:Q", title=f"Mean {feature_n}"),
        tooltip=["month:N", "count:Q", "mean:Q", "median:Q"]
    )
    line_median = base.mark_line(point=True, color="#ff7f0e").encode(
        y=alt.Y("median:Q", title=f"Median {feature_n}")
    )

    (line_mean + line_median).resolve_scale(y="shared")
else:
    print("Need both feature_n (money) and feature_2 (date) for monetary trend over time.")

# B.4 Extra 6: Compare monetary value by status/category (B.2 feature_1)
if feature_n and "feature_1" in globals() and feature_1:
    tmp = df[[feature_1, feature_n]].copy()
    tmp[feature_1] = tmp[feature_1].fillna("<<MISSING>>").astype(str)

    grp = (
        tmp.dropna(subset=[feature_n])
           .groupby(feature_1)[feature_n]
           .agg(count="count", mean="mean", median="median")
           .reset_index()
           .sort_values("median", ascending=False)
    )
    grp["mean"] = grp["mean"].round(2)
    grp["median"] = grp["median"].round(2)
    display(grp)

    alt.Chart(grp).mark_bar().encode(
        x=alt.X("median:Q", title=f"Median {feature_n}"),
        y=alt.Y(f"{feature_1}:N", sort="-x", title=feature_1),
        tooltip=[feature_1, "count", "mean", "median"]
    ).properties(title=f"Median {feature_n} by {feature_1}", width=750, height=300)
else:
    print("Need feature_n and feature_1 (status/category) to compare monetary value by status.")

,month,mean,median,count
0,2011-05,12391.81,6301.63,47
1,2011-06,10342.46,3953.99,217
2,2011-07,8690.74,3953.99,215
3,2011-08,7848.03,3953.99,189
4,2011-09,12734.86,3953.99,250
5,2011-10,10644.08,3953.99,239
6,2011-11,6004.92,3953.99,267
7,2011-12,10474.69,3953.99,268
8,2012-01,8979.03,3953.99,259
9,2012-02,7628.46,3953.99,214


,month,mean,median,count
23,2013-04,6670.39,2264.25,426
24,2013-05,8422.33,2288.92,436
25,2013-06,7702.00,2560.25,741
26,2013-07,3160.34,82.85,1755
27,2013-08,2086.50,86.72,1785
28,2013-09,2834.72,98.85,1796
29,2013-10,2740.69,138.11,1971
30,2013-11,1748.37,596.69,2100
31,2013-12,2225.39,136.98,2053
32,2014-01,2239.34,166.44,2136


,status,count,mean,median
0,5.0,31465,3916.0,865.2


In [86]:
# B.4 Extra 7: Consistency check if component totals exist
# Checks if SubTotal + TaxAmt + Freight ~= TotalDue
cands = {c.lower(): c for c in df.columns}

sub_col = next((cands[k] for k in cands if k in ["subtotal", "sub_total"]), None)
tax_col = next((cands[k] for k in cands if "tax" in k and "amt" in k or k == "taxamt"), None)
freight_col = next((cands[k] for k in cands if "freight" in k), None)
total_col = next((cands[k] for k in cands if k in ["totaldue", "total_due"]), None)

print("Detected components:", {"sub_total": sub_col, "tax": tax_col, "freight": freight_col, "total_due": total_col})

if sub_col and tax_col and freight_col and total_col:
    tmp = df[[sub_col, tax_col, freight_col, total_col]].dropna().copy()
    tmp["recalc_total"] = tmp[sub_col] + tmp[tax_col] + tmp[freight_col]
    tmp["abs_diff"] = (tmp["recalc_total"] - tmp[total_col]).abs()

    display(tmp["abs_diff"].describe())

    # Flag large inconsistencies
    bad = tmp[tmp["abs_diff"] > 0.01].sort_values("abs_diff", ascending=False).head(15)
    print("Top inconsistencies (abs diff > 0.01):")
    display(bad)
else:
    print("Component total columns not all present; skipping totals consistency check.")

Detected components: {'sub_total': 'sub_total', 'tax': None, 'freight': 'freight', 'total_due': 'total_due'}
Component total columns not all present; skipping totals consistency check.


In [87]:
feature_n_insights = f"""
Feature explored (B.4): {feature_n} (order-level monetary value)

Why this feature matters
- {feature_n} represents the monetary magnitude of each order and is central for measuring customer value and engagement.
- For Student A’s classification goal (predicting whether a customer will purchase again), monetary behaviour is often highly predictive when aggregated at customer level (e.g., total spend, average order value, max order value, spend in last 90 days).

Distribution and key patterns observed
- The histogram and summary statistics indicate that {feature_n} is typically right-skewed (many smaller orders with fewer high-value orders).
- A boxplot highlights the presence of extreme values; these may be legitimate bulk purchases or may reflect data errors depending on business rules.
- Viewing the top 10 orders provides concrete examples of the extreme tail and helps decide whether outliers require treatment.

Data quality checks performed (HD-level)
- Completeness: measured missingness rate for {feature_n}.
- Validity: checked for negative values and zero values (if present), which may indicate cancellations/returns, adjustments, or data issues.
- Outliers: applied an IQR-based rule to identify unusually high/low values.
- Visualisation improvement: a log1p({feature_n}) transformation was used to better visualise the bulk of the distribution under heavy skew.
- Temporal stability (if order date was available): analysed monthly mean/median {feature_n} to detect changes over time (seasonality or structural shifts).
- Operational linkage (if status feature is available): compared median {feature_n} across order status categories to see whether cancelled/exception orders differ in monetary patterns.

Limitations and issues to consider
- Skew and outliers can distort mean-based features; using robust statistics (median, trimmed mean) and/or log transforms is recommended.
- If cancelled orders are included, {feature_n} may not represent realised revenue; business logic should define whether to exclude cancelled orders or model them separately.
- If the “total” field is composed of components (e.g., SubTotal + Tax + Freight), inconsistencies should be checked to identify potential ETL or rounding issues.

Recommended handling for modelling (Student A)
- Aggregate to customer-level features with time awareness:
  - total_spend, avg_order_value, median_order_value, max_order_value,
  - spend_last_30/90/180 days, and change in spend over time.
- Consider transforming {feature_n} using log1p for models sensitive to skew.
- Treat extreme outliers carefully:
  - confirm whether they are legitimate bulk orders,
  - if necessary, cap/winsorise to reduce undue influence while preserving signal.
- Ensure leakage control: only use monetary values from orders that occurred before the prediction “as-of” date.

Overall conclusion
- {feature_n} is a high-value feature for behavioural modelling because it captures purchase intensity and supports strong customer-level monetary predictors, provided skew/outliers and cancelled-order logic are handled appropriately.
"""

In [88]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

In [89]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(feature_n_insights.strip()) + \
            "</pre>"

display(HTML(html_text))

In [90]:
overall_insights = f"""
Overall EDA insights (sales_order_header.csv)

1) Dataset readiness and structure
- The sales_order_header table is an order-level transactional dataset (one row per order header) that is well-suited for behavioural feature engineering.
- It contains the essential join keys (e.g., customer_id and an order ID) required to aggregate orders to the customer level and to link with other tables (customer, order_detail).

2) Most important findings from selected features
B.2 (status)
- Status captures the operational outcome/lifecycle of orders and is crucial for defining what counts as a “valid purchase”.
- It should primarily be used for data cleaning and feature engineering (e.g., cancellation rate per customer) rather than being used directly in a way that introduces leakage.

B.3 (order_date / time reference)
- The order date provides the timeline needed to build recency, frequency, and trend-based behavioural features.
- Time-based analysis (monthly volume, weekday patterns, seasonality) helps validate consistent data capture and supports calendar-based feature creation.
- Correct handling of time is critical to avoid leakage (features must be computed only from information available before the prediction point).

B.4 ({feature_n} monetary value)
- Order monetary value is a strong behavioural signal but typically shows heavy right-skew and outliers.
- Robust handling is required (median-based aggregates, log transforms, outlier investigation/capping).
- Monetary trends over time and differences by status can reveal operational effects (e.g., cancelled/exception orders) and guide feature construction.

3) Key data quality and modelling risks (to document)
- Leakage risk: fields only known after fulfilment (final status, ship-related timestamps) must not be used incorrectly when predicting future behaviour.
- Missingness and inconsistent formats: date parsing failures or missing keys reduce usable records and can bias results if missingness is systematic.
- Outliers in monetary values: can dominate averages and mislead models unless treated robustly.

4) Recommended next step for Student A (classification pipeline)
- Build a customer-level modelling table using a time-based “as-of” date framework:
  - recency_days (days since last order),
  - frequency counts in rolling windows (30/90/180/365 days),
  - monetary aggregates (total/mean/median/max spend in windows),
  - engineered operational metrics (e.g., cancellation rate from status).
- Use a chronological train/test split to evaluate the model realistically and ensure generalisation to future periods.

Conclusion
- Overall, the dataset is appropriate for Student A’s classification task, and the chosen B.2/B.3/B.4 features collectively support a strong, leakage-aware behavioural modelling approach.
"""

In [91]:
from IPython.display import display, HTML
import html

html_text = "<pre style='white-space: pre-wrap; word-wrap: break-word; font-size: 13px; line-height: 1.35;'>" \
            + html.escape(overall_insights.strip()) + \
            "</pre>"

display(HTML(html_text))